In [ ]:
import pandas as pd
from perturbation_prediction_metrics import PerturbationPredictionMetrics
from predictors import build

from data import prepare_data

In [ ]:
from config import get_config

SPLITS = "/workspace/experiments/05152026_cellbox_comeback/splits"

config = get_config()
# config.model_name = "cellbox_metabolite"
# config.experiment_subset = "all"
# config.adata_path = (
#     "/workspace/data/05142026_metabolome/adata_de122_lce75_merged.metabolome.h5ad"
# )
# config.perturbation_col = "top_target_unthresholded"

# config.split.train_extra_targets_path = f"{SPLITS}/non_tfs.txt"

# config.models.cellbox_metabolite.filter_regulators = True
# # config.models.cellbox_metabolite.training.n_epochs = 500
# config.models.cellbox_metabolite.training.validate_every_n_epochs = 0

# config.models.cellbox_metabolite.training.n_val_steps = 5
# config.models.cellbox_metabolite.training.train_mode = "rollout"
# config.models.cellbox_metabolite.training.n_train_steps = 5

# config.models.cellbox_metabolite.training.early_stopping_metric = "reco_loss"
# config.models.cellbox_metabolite.training.early_stopping_patience = 10
# config.models.cellbox_metabolite.training.early_stopping_mode = "min"

config.model_name = "cellbox"
config.experiment_subset = "all"
config.adata_path = (
    "/workspace/data/05142026_metabolome/adata_de122_lce75_merged.metabolome.h5ad"
)
config.perturbation_col = "top_target_unthresholded"
config.normalization.target_sum = 1e4

config.split.train_extra_targets_path = f"{SPLITS}/non_tfs.txt"

config.models.cellbox.filter_regulators = True
config.models.cellbox.training.validate_every_n_epochs = 0

config.models.cellbox.training.n_val_steps = 5
config.models.cellbox.training.train_mode = "rollout"
config.models.cellbox.training.n_train_steps = 5

config.models.cellbox.training.early_stopping_metric = "reco_loss"
config.models.cellbox.training.early_stopping_patience = 10000
config.models.cellbox.training.early_stopping_mode = "min"

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(config.adata_path)
sc.pp.normalize_total(adata, target_sum=config.normalization.target_sum)
sc.pp.log1p(adata)
adata1 = adata[adata.obs["experiment"] == "de122"].copy()
adata2 = adata[adata.obs["experiment"] == "lce75"].copy()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from tqdm import tqdm


def compute_lfc(ad):
    ctrl = np.asarray(ad[ad.obs["target"] == "nontargeting"].X.mean(axis=0)).ravel()
    targets = ad.obs["target"].unique()
    return {
        t: np.asarray(ad[ad.obs["target"] == t].X.mean(axis=0)).ravel() - ctrl
        for t in tqdm(targets)
    }


lfc1 = compute_lfc(adata1)
lfc2 = compute_lfc(adata2)

targets = [t for t in lfc1 if t in lfc2]
df = pd.DataFrame(
    {"pearson": [pearsonr(lfc1[t], lfc2[t])[0] for t in targets]},
    index=pd.Index(targets, name="target"),
)

In [ ]:
df["pearson"].sort_values()

In [ ]:
df["pearson"].loc["emrE"]

In [ ]:
df.sort_values("pearson", ascending=False).hist(bins=50)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(lfc1["nusG"], lfc2["nusG"])

In [ ]:
import matplotlib.pyplot as plt

adata_train.X.toarray().std(0).min()

In [ ]:
adata_train, adata_test, adata_control, Amask = prepare_data(config)
pred = build(config.model_name, config, adata_train, Amask)
pred.fit()

In [ ]:
result = pred.collect_predictions(adata_test, adata_control, config.perturbation_col)

In [ ]:
result = pred.collect_predictions(adata_test, adata_control, config.perturbation_col)
result["model_name"] = config.tag
metrics = PerturbationPredictionMetrics().from_result(
    result, output_path=config.output_path
)

In [ ]:
print(metrics.mean(numeric_only=True))

In [ ]:
# valid_train_perts = adata_train.obs[config.perturbation_col].isin(adata_train.var_names)
# result_train = pred.collect_predictions(
#     adata_train[valid_train_perts],
#     adata_control,
#     config.perturbation_col,
# )
# result_train["model_name"] = "train"
# metrics_train = PerturbationPredictionMetrics().from_result(
#     result_train, output_path=config.output_path
# )
# print(metrics_train.mean(numeric_only=True))

One batch only

```
n_cells_gt               7.386364
n_cells_pred            95.000000
mu_mse                   0.017717
mu_pearson               0.978017
lfc_mse                  0.017717
lfc_pearson              0.064582
mu_mse_top_degs          0.483475
mu_pearson_top_degs      0.490490
lfc_mse_top_degs         0.483475
lfc_pearson_top_degs     0.300232
dtype: float64
```


all batches

```
n_cells_gt               16.090909
n_cells_pred            203.000000
mu_mse                    0.007131
mu_pearson                0.991133
lfc_mse                   0.007131
lfc_pearson               0.061696
mu_mse_top_degs           0.270563
mu_pearson_top_degs       0.727959
lfc_mse_top_degs          0.270563
lfc_pearson_top_degs      0.256134
```


In [ ]:
pred.estimator.epoch_history_df["val_loss"].plot()

In [ ]:
pred.estimator.state.params["gate_net"]["kernel"]

In [ ]:
pred.estimator.state.params["gate_net"]

In [ ]:
Amat = pred.estimator.get_Amat()
is_tf = Amat.sum(0) != 0
tf_names = Amat.columns[is_tf]
tf_names

In [ ]:
tf_reps = pred.estimator.state.params["gate_net"]["kernel"][:, is_tf.values].T
tf_reps.shape

In [ ]:
import matplotlib.pyplot as plt

plt.hist(tf_reps.ravel(), bins=100)

In [ ]:
import seaborn as sns

sns.clustermap(
    tf_reps,
)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# tf_reps_pca = PCA(n_components=50).fit_transform(tf_reps)
# tf_reps_tsne = TSNE(n_components=2, perplexity=50).fit_transform(tf_reps)
tf_reps_tsne = PCA(n_components=2).fit_transform(tf_reps)

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
tf_metadata = (
    pd.read_excel(
        "/workspace/experiments/05152026_cellbox_comeback/1-s2.0-S2452310021000998-mmc1.xlsx",
        sheet_name="TFs",
    ).assign(tf_lower=lambda df: df["Transcription factor"].str.lower())
    # .set_index("tf_lower")
)
tf_metadata.effector_name.value_counts()

# tf_metadata = tf_metadata.loc[
#     lambda x: x["effector_name"].isin(["Zn", "pyruvate", "ATP", "Cu", "Cd"])
# ]

In [ ]:
effectors_to_keep = [
    "Zn",
    "pyruvate",
    "ATP",
    "Cu",
    "Cd",
    "Ni",
    "glyoxylate",
    "D-galactose",
    "Thiosulphate",
    "L-arginine",
    "L-tryptophan",
    "D-galacturonate",
    "L-rhamnose",
    "Mn",
    "[2Fe-2S] oxidized",
    "allantoin",
    "2,4-dinitrophenol",
    "D-glucuronate",
    "Co",
    "[2Fe-2S] reduced",
    "Fe",
]

In [ ]:
tf_metadata = tf_metadata.loc[lambda x: x["effector_name"].isin(effectors_to_keep)]

In [ ]:
import plotnine as gg

plot_df = pd.DataFrame(tf_reps_tsne, columns=["tsne1", "tsne2"])
plot_df["tf_name"] = tf_names.values
plot_df["tf_lower"] = plot_df["tf_name"].str.lower()
plot_df = plot_df.merge(tf_metadata, on="tf_lower", how="left")
# plot_df.effector_name.fillna("other", inplace=True)

(
    gg.ggplot(plot_df, gg.aes(x="tsne1", y="tsne2", color="effector_name"))
    + gg.geom_point()
    + gg.theme_bw()
    + gg.theme(legend_position="none")
)

In [ ]:
import plotly.express as px

px.scatter(plot_df, x="tsne1", y="tsne2", color="effector_name", hover_name="tf_name")